# Failure analysis
Manual coding workspace for the failure taxonomy. Read actual responses; do not rely on category counts alone.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from reliability_lab.pipeline.result_collector import read_raw, load_scored
from reliability_lab.evaluation.metrics import compute_metrics
RESULTS = "results"   # or "results_smoke" (MOCK data: pipeline testing only)

from reliability_lab.evaluation.failures import add_failure_categories

In [ ]:
scored = add_failure_categories(load_scored(RESULTS))
fails = scored[scored.failure_category.notna()]
fails.groupby(['failure_category','model_key']).size().unstack(fill_value=0)

## Read examples per category

In [ ]:
CATEGORY = fails.failure_category.value_counts().index[0] if len(fails) else None
for _, r in fails[fails.failure_category==CATEGORY].head(5).iterrows():
    print(f"[{r.model_key}] {r.item_id}
{r.response}
{'-'*60}")

## Rows flagged for human review
Fill `label` (pass/fail) in `results/processed/review_queue.csv`, then save lines `{"item_id":..., "model_key":..., "repeat":..., "label":"pass"}` to `results/human_labels.jsonl`.

In [ ]:
q = pd.read_csv(f'{RESULTS}/processed/review_queue.csv')
print(len(q), 'rows queued')
q.head(10)

## Automatic vs. human agreement

In [ ]:
from reliability_lab.evaluation.evaluators import load_human_labels, apply_human_labels, human_agreement
lab = apply_human_labels(scored, load_human_labels(ROOT/'results'/'human_labels.jsonl'))
human_agreement(lab)